# ADS ViT Forensics — Colab Quickstart

End-to-end Colab workflow for regenerating JSON artifacts and figures for

**Attention Divergence Score: A Forensic Metric for Characterizing Parameter-Level Attacks in Vision Transformers**.

This notebook is intentionally more operational than the README. The README documents the repository and gives minimal CLI examples; this notebook contains the full Drive/path setup and the complete experiment sequence.


## 0. Runtime and repository setup

Use a GPU runtime for experiment generation. For verification-only runs, GPU is not required.

Recommended sequence:

1. Mount Google Drive.
2. Clone or update the GitHub repository.
3. Install dependencies.
4. Edit the path configuration cell.
5. Run only the experiment sections you need.


In [ ]:
# GPU sanity check
import os
import sys
import subprocess
from pathlib import Path

try:
    import torch
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
        print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
except Exception as exc:
    print("PyTorch not available yet:", exc)

subprocess.run("nvidia-smi || true", shell=True, check=False)


In [ ]:
from google.colab import drive

drive.mount('/content/drive')


In [ ]:
# Clone or update the repository.
# The GitHub repo is separate from the Google Drive checkpoint folder.
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/djokobandjur/ads-vit-forensics.git"
REPO_DIR = Path("/content/ads-vit-forensics")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)

os.chdir(REPO_DIR)
print("Repository:", REPO_DIR)
print("HEAD:")
subprocess.run(["git", "-C", str(REPO_DIR), "log", "-1", "--oneline"], check=False)


In [ ]:
# Install repository dependencies.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")], check=True)
print("Dependencies installed.")


## 1. Path configuration

Edit this cell once. The checkpoint layout mirrors the Drive organization:

```text
/content/drive/MyDrive/ads-vit-forensics/
├── ImageNet100/
├── CIFAR100/
└── CIFAR100_canonical/
```

Do not pass the parent checkpoint root to an experiment script. Pass the dataset-specific folder (`ImageNet100`, `CIFAR100`, or `CIFAR100_canonical`).


In [ ]:
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive")

# Google Drive folder that contains checkpoint folders.
CHECKPOINT_ROOT = DRIVE_ROOT / "ads-vit-forensics"
IMAGENET_MODELS_DIR = CHECKPOINT_ROOT / "ImageNet100"
CIFAR_MODELS_DIR = CHECKPOINT_ROOT / "CIFAR100"
CANONICAL_CIFAR_MODELS_DIR = CHECKPOINT_ROOT / "CIFAR100_canonical"

# Drive output location for newly generated JSONs, figures, and logs.
# These are generated artifacts; after verification, copy the needed files into the GitHub repo tree.
RUN_DIR = CHECKPOINT_ROOT / "generated_outputs"
DATA_DIR = RUN_DIR / "data"
ROBUSTNESS_DATA_DIR = DATA_DIR / "robustness"
OUTPUT_DIR = RUN_DIR              # generate_ads_figures.py writes OUTPUT_DIR/figures/
LOG_DIR = RUN_DIR / "logs"

# ImageNet-100 validation data. ImageNet is not redistributed.
# Option A: set IMAGENET100_VAL_DIR directly if the val/ folder already exists.
# Option B: set IMAGENET100_TAR to a private tarball that expands to imagenet100_resized/{train,val}.
IMAGENET100_TAR = CHECKPOINT_ROOT / "datasets" / "imagenet100_resized.tar"
IMAGENET100_ROOT = Path("/content/imagenet100_resized")
IMAGENET100_VAL_DIR = IMAGENET100_ROOT / "val"

# CIFAR-100 cache. The CIFAR scripts can download CIFAR-100 through torchvision.
CIFAR100_CACHE = Path("/tmp/cifar100")

PRIMARY_SEEDS = [42, 123, 456, 789, 1011, 1213]
CANONICAL_SEEDS = [1, 5, 7, 11, 13, 21, 42, 99, 123, 456, 2024, 31337]

for p in [DATA_DIR, ROBUSTNESS_DATA_DIR, OUTPUT_DIR, LOG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Repository clone:", REPO_DIR)
print("Checkpoint root:", CHECKPOINT_ROOT)
print("ImageNet models:", IMAGENET_MODELS_DIR)
print("CIFAR models:", CIFAR_MODELS_DIR)
print("Canonical CIFAR models:", CANONICAL_CIFAR_MODELS_DIR)
print("Run data:", DATA_DIR)
print("Robustness data:", ROBUSTNESS_DATA_DIR)
print("Output dir:", OUTPUT_DIR)
print("Log dir:", LOG_DIR)


## 2. Helper functions

The helpers below run repository scripts and tee stdout/stderr to Drive logs.


In [ ]:
import shlex
import subprocess
from datetime import datetime

SCRIPTS_DIR = REPO_DIR / "scripts"
ROBUSTNESS_SCRIPTS_DIR = SCRIPTS_DIR / "robustness"


def run_logged(args, log_name, cwd=REPO_DIR):
    """Run a command and tee stdout/stderr to a log file in LOG_DIR."""
    args = [str(a) for a in args]
    log_path = LOG_DIR / log_name
    print("\n$", " ".join(shlex.quote(a) for a in args))
    print("Log:", log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with open(log_path, "w", encoding="utf-8") as log:
        log.write(f"# Started: {datetime.now().isoformat()}\n")
        log.write("$ " + " ".join(shlex.quote(a) for a in args) + "\n\n")
        proc = subprocess.Popen(
            args,
            cwd=str(cwd),
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        for line in proc.stdout:
            print(line, end="")
            log.write(line)
        ret = proc.wait()
        log.write(f"\n# Finished: {datetime.now().isoformat()}  returncode={ret}\n")
    if ret != 0:
        raise RuntimeError(f"Command failed with return code {ret}: {' '.join(args)}")
    return log_path


def py_script(script_relative_path, *script_args, log_name=None):
    """Run a Python script from the repository using a relative path such as scripts/ads_experiment.py."""
    script_path = REPO_DIR / script_relative_path
    if not script_path.exists():
        raise FileNotFoundError(f"Script not found: {script_path}")
    if log_name is None:
        log_name = Path(script_relative_path).name.replace('.py', '.log')
    return run_logged([sys.executable, "-u", str(script_path), *script_args], log_name)


## 3. Dataset preparation

ImageNet-100 is not redistributed. This cell extracts a private ImageNet-100 tarball from Drive if the validation directory is not already present.

CIFAR-100 is handled by the CIFAR scripts through `torchvision.datasets.CIFAR100(download=True)`.


In [ ]:
import shutil

if IMAGENET100_VAL_DIR.exists():
    print("ImageNet-100 validation directory already exists:", IMAGENET100_VAL_DIR)
else:
    if not IMAGENET100_TAR.exists():
        print("ImageNet-100 tar not found:", IMAGENET100_TAR)
        print("Set IMAGENET100_TAR or prepare IMAGENET100_VAL_DIR manually before running ImageNet scripts.")
    else:
        print("Extracting:", IMAGENET100_TAR)
        shutil.rmtree(IMAGENET100_ROOT, ignore_errors=True)
        IMAGENET100_ROOT.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(["tar", "-xf", str(IMAGENET100_TAR), "-C", "/content"], check=True)
        print("ImageNet val:", IMAGENET100_VAL_DIR, "exists=", IMAGENET100_VAL_DIR.exists())

CIFAR100_CACHE.mkdir(parents=True, exist_ok=True)
print("CIFAR-100 cache:", CIFAR100_CACHE)


## 4. Reference indices

The paper uses fixed reference-image indices for exact ADS reference-set reproduction. The repository ships these under `data/`.


In [ ]:
import json
import shutil

for name in ["ads_ref_indices.json", "ads_ref_indices_imagenet100.json", "ads_ref_indices_cifar100.json"]:
    repo_ref = REPO_DIR / "data" / name
    run_ref = DATA_DIR / name
    if repo_ref.exists() and not run_ref.exists():
        shutil.copy2(repo_ref, run_ref)
        print("Copied:", name)
    elif run_ref.exists():
        print("Already present:", name)

main_ref = DATA_DIR / "ads_ref_indices.json"
if main_ref.exists():
    ref = json.loads(main_ref.read_text())
    print("ads_ref_indices.json:", len(ref), "indices; unique=", len(set(ref)) == len(ref))
else:
    print("ads_ref_indices.json not found. Some scripts may regenerate it with their fixed seed.")


## 5. Main PE-only ADS sweeps

Outputs:

- `ads_results.json`
- `ads_results_cifar100.json`

In [ ]:
py_script(
    'scripts/ads_experiment.py',
    '--models_dir', IMAGENET_MODELS_DIR,
    '--val_dir', IMAGENET100_VAL_DIR,
    '--output_path', DATA_DIR / "ads_results.json",
    log_name='ads_experiment_imagenet.log',
)


In [ ]:
py_script(
    'scripts/ads_experiment_cifar.py',
    '--models_dir', CIFAR_MODELS_DIR,
    '--val_dir', CIFAR100_CACHE,
    '--output_path', DATA_DIR / "ads_results_cifar100.json",
    log_name='ads_experiment_cifar100.log',
)


## 6. Specificity sweeps: PE, QKV, MLP, all-weights

Outputs:

- `ads_specificity.json`
- `ads_specificity_cifar.json`

In [ ]:
py_script(
    'scripts/ads_specificity.py',
    '--models_dir', IMAGENET_MODELS_DIR,
    '--val_dir', IMAGENET100_VAL_DIR,
    '--output_path', DATA_DIR / "ads_specificity.json",
    log_name='ads_specificity_imagenet.log',
)


In [ ]:
py_script(
    'scripts/ads_specificity_cifar.py',
    '--models_dir', CIFAR_MODELS_DIR,
    '--val_dir', CIFAR100_CACHE,
    '--output_path', DATA_DIR / "ads_specificity_cifar.json",
    log_name='ads_specificity_cifar100.log',
)


## 7. ROC analysis against benign shifts

Output: `ads_roc_v2.json`.

In [ ]:
py_script(
    'scripts/ads_roc_analysis.py',
    '--models_dir', IMAGENET_MODELS_DIR,
    '--val_dir', IMAGENET100_VAL_DIR,
    '--output_path', DATA_DIR / "ads_roc_v2.json",
    log_name='ads_roc_analysis.log',
)


## 8. Adaptive and reference-set evasion analyses

Outputs:

- `ads_adaptive.json`
- `ads_ref_evasion.json`

In [ ]:
py_script(
    'scripts/ads_adaptive.py',
    '--models_dir', IMAGENET_MODELS_DIR,
    '--val_dir', IMAGENET100_VAL_DIR,
    '--output_path', DATA_DIR / "ads_adaptive.json",
    log_name='ads_adaptive.log',
)


In [ ]:
py_script(
    'scripts/ads_ref_evasion.py',
    '--models_dir', IMAGENET_MODELS_DIR,
    '--val_dir', IMAGENET100_VAL_DIR,
    '--output_path', DATA_DIR / "ads_ref_evasion.json",
    log_name='ads_ref_evasion.log',
)


## 9. Fine-grid threshold calibration

Output: `ads_threshold_fine.json`.

In [ ]:
py_script(
    'scripts/ads_threshold_fine.py',
    '--models_dir', IMAGENET_MODELS_DIR,
    '--val_dir', IMAGENET100_VAL_DIR,
    '--output_path', DATA_DIR / "ads_threshold_fine.json",
    log_name='ads_threshold_fine.log',
)


## 10. Residual-stream probing

Outputs:

- `ads_probing_residual.json`
- `ads_probing_residual_cifar.json`

In [ ]:
py_script(
    'scripts/ads_probing_residual.py',
    '--models_dir', IMAGENET_MODELS_DIR,
    '--val_dir', IMAGENET100_VAL_DIR,
    '--output_path', DATA_DIR / "ads_probing_residual.json",
    log_name='ads_probing_residual_imagenet.log',
)


In [ ]:
py_script(
    'scripts/ads_probing_residual_cifar.py',
    '--models_dir', CIFAR_MODELS_DIR,
    '--val_dir', CIFAR100_CACHE,
    '--output_path', DATA_DIR / "ads_probing_residual_cifar.json",
    log_name='ads_probing_residual_cifar100.log',
)


## 11. Shared-δ / per-buffer comparison

Output: `ads_comparison.json`.

In [ ]:
py_script(
    'scripts/ads_comparison.py',
    '--models_dir', IMAGENET_MODELS_DIR,
    '--val_dir', IMAGENET100_VAL_DIR,
    '--output_path', DATA_DIR / "ads_comparison.json",
    log_name='ads_comparison.log',
)


## 12. Optional: canonical CIFAR-100 n=12 protocol-robustness runs

These scripts are auxiliary and support the independent canonical CIFAR-100 n=12 robustness cohort. They are not part of the primary n=6 sweep.

Outputs are written to `DATA_DIR/robustness/`.


In [ ]:
py_script(
    'scripts/robustness/ads_experiment_cifar_protocol_robustness_n12.py',
    "--models_dir", CANONICAL_CIFAR_MODELS_DIR,
    "--val_dir", CIFAR100_CACHE,
    "--output_path", ROBUSTNESS_DATA_DIR / 'ads_results_cifar100_canonical_n12.json',
    log_name='robustness_ads_experiment_cifar_n12.log',
)


In [ ]:
py_script(
    'scripts/robustness/ads_specificity_cifar_protocol_robustness_n12.py',
    "--models_dir", CANONICAL_CIFAR_MODELS_DIR,
    "--val_dir", CIFAR100_CACHE,
    "--output_path", ROBUSTNESS_DATA_DIR / 'ads_specificity_cifar100_canonical_n12.json',
    log_name='robustness_ads_specificity_cifar_n12.log',
)


In [ ]:
py_script(
    'scripts/robustness/ads_probing_residual_cifar_protocol_robustness_n12.py',
    "--models_dir", CANONICAL_CIFAR_MODELS_DIR,
    "--val_dir", CIFAR100_CACHE,
    "--output_path", ROBUSTNESS_DATA_DIR / 'ads_probing_residual_cifar100_canonical_n12.json',
    log_name='robustness_ads_probing_residual_cifar_n12.log',
)


## 13. Generate paper figures

This uses `scripts/generate_ads_figures.py`, including the Fig. 2 log-scale zero-column fix.


In [ ]:
py_script(
    "scripts/generate_ads_figures.py",
    "--data-dir", DATA_DIR,
    "--output-dir", OUTPUT_DIR,
    log_name="generate_ads_figures.log",
)

fig_dir = OUTPUT_DIR / "figures"
print("\nGenerated figures:")
if fig_dir.exists():
    for p in sorted(fig_dir.glob("ads_fig*")):
        print(" ", p)
else:
    print("Figure directory not found:", fig_dir)


## 14. CPU-only verification

Run this after the JSON files are present. It does not require GPU or model checkpoints.


In [ ]:
py_script(
    "scripts/reproduce.py",
    "--data-dir", DATA_DIR,
    "--output-dir", OUTPUT_DIR,
    "--no-figures",
    log_name="reproduce.log",
)


## 15. Optional: sync generated artifacts into the GitHub repo tree

Use this only after checking the run outputs. It copies JSONs to `repo/data/`, robustness JSONs to `repo/data/robustness/`, and figures to `repo/figures/` so they can be committed.


In [ ]:
import shutil

REPO_DATA_DIR = REPO_DIR / "data"
REPO_ROBUSTNESS_DIR = REPO_DATA_DIR / "robustness"
REPO_FIGURES_DIR = REPO_DIR / "figures"

REPO_DATA_DIR.mkdir(parents=True, exist_ok=True)
REPO_ROBUSTNESS_DIR.mkdir(parents=True, exist_ok=True)
REPO_FIGURES_DIR.mkdir(parents=True, exist_ok=True)

for p in DATA_DIR.glob("*.json"):
    shutil.copy2(p, REPO_DATA_DIR / p.name)
    print("copied data:", p.name)

if ROBUSTNESS_DATA_DIR.exists():
    for p in ROBUSTNESS_DATA_DIR.glob("*.json"):
        shutil.copy2(p, REPO_ROBUSTNESS_DIR / p.name)
        print("copied robustness:", p.name)

fig_dir = OUTPUT_DIR / "figures"
if fig_dir.exists():
    for p in fig_dir.glob("ads_fig*.png"):
        shutil.copy2(p, REPO_FIGURES_DIR / p.name)
        print("copied figure:", p.name)
    for p in fig_dir.glob("ads_fig*.pdf"):
        shutil.copy2(p, REPO_FIGURES_DIR / p.name)
        print("copied figure:", p.name)

print("\nCheck git status:")
subprocess.run(["git", "-C", str(REPO_DIR), "status", "--short"], check=False)


## 16. Final artifact inventory and optional zip

Use this before uploading artifacts to GitHub or archiving a generated run.


In [ ]:
import zipfile

print("JSON files:")
for p in sorted(DATA_DIR.glob("*.json")):
    print(f"  {p.name:45s} {p.stat().st_size/1024/1024:8.2f} MB")

print("\nRobustness JSON files:")
if ROBUSTNESS_DATA_DIR.exists():
    for p in sorted(ROBUSTNESS_DATA_DIR.glob("*.json")):
        print(f"  {p.name:45s} {p.stat().st_size/1024/1024:8.2f} MB")

print("\nFigure files:")
fig_dir = OUTPUT_DIR / "figures"
if fig_dir.exists():
    for p in sorted(fig_dir.glob("ads_fig*")):
        print(f"  {p.name:45s} {p.stat().st_size/1024/1024:8.2f} MB")

zip_path = RUN_DIR / "ads_tifs_generated_outputs.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for folder in [DATA_DIR, OUTPUT_DIR / "figures", LOG_DIR]:
        if folder.exists():
            for p in folder.rglob("*"):
                if p.is_file():
                    zf.write(p, p.relative_to(RUN_DIR))

print("\nCreated zip:", zip_path)


## Notes

- For routine repository checks, run `scripts/reproduce.py` against the shipped JSON files; no GPU is required.
- Keep `data/ads_ref_indices.json` fixed to reproduce the paper's reference-set measurements exactly.
- Generated ImageNet images are not redistributed; only JSON results, figures, code, and fixed reference indices belong in the public repository.
- The canonical CIFAR-100 n=12 robustness cohort is auxiliary and should remain under `data/robustness/` and `scripts/robustness/`.
